# 📊 Exploratory Data Analysis — ChurnGuard AI

Dataset: **BigML Telecom Churn** (pre-split 80/20).

This notebook explores the customer base to understand churn drivers before modelling. Production logic lives in `src/`; this notebook is for experimentation only.

> **Note on column mapping.** The BigML dataset has no `Contract`, `MonthlyCharges` or `tenure` columns (those belong to IBM Telco). We use the natural BigML equivalents:
> - contract type → **International plan** (the committed plan)
> - monthly charges → **total_charge** (engineered: day+eve+night+intl charge)
> - tenure → **Account length**

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
plt.rcParams['figure.figsize'] = (9, 5)

from src.data_preprocessing import load_clean_split
from src.feature_engineering import add_features

# Combine the 80/20 splits for a single EDA view; add engineered features
# (we need total_charge for the spend analysis).
train, test = load_clean_split()
df = add_features(pd.concat([train, test], ignore_index=True))
print(f'Train: {train.shape} | Test: {test.shape} | Combined: {df.shape}')
df.head()

## 1. Structure & data quality

In [ ]:
df.info()

In [ ]:
print('Missing values:', int(df.isna().sum().sum()))
print('Duplicate rows:', int(df.duplicated().sum()))
df.describe().T

## 2. Target distribution & class imbalance

`Churn` is 1 = churned, 0 = retained.

In [ ]:
print(f'Overall churn rate: {df["Churn"].mean():.1%}')
print(df['Churn'].value_counts())

## 3. Requested churn-driver charts

The four charts asked for, using the BigML column mapping above:
1. **Churn distribution**
2. **Contract type (International plan) vs churn**
3. **Monthly charges (total_charge) vs churn**
4. **Tenure (Account length) vs churn**

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# (1) Churn distribution
df['Churn'].map({0: 'Retained', 1: 'Churned'}).value_counts().plot(
    kind='bar', color=['#16a34a', '#dc2626'], ax=axes[0, 0])
axes[0, 0].set_title('1) Churn distribution')
axes[0, 0].set_ylabel('Customers'); axes[0, 0].tick_params(axis='x', rotation=0)

# (2) Contract type -> International plan vs churn
(df.groupby('International plan')['Churn'].mean() * 100).plot(
    kind='bar', color='#2563eb', ax=axes[0, 1])
axes[0, 1].set_title('2) International plan vs churn')
axes[0, 1].set_ylabel('Churn rate (%)'); axes[0, 1].tick_params(axis='x', rotation=0)

# (3) Monthly charges -> total_charge vs churn
for label, color in [(0, '#16a34a'), (1, '#dc2626')]:
    axes[1, 0].hist(df[df['Churn'] == label]['total_charge'], bins=30, alpha=0.55,
                    color=color, label='Churned' if label else 'Retained')
axes[1, 0].set_title('3) total_charge vs churn')
axes[1, 0].set_xlabel('total_charge'); axes[1, 0].legend()

# (4) Tenure -> Account length vs churn
for label, color in [(0, '#16a34a'), (1, '#dc2626')]:
    axes[1, 1].hist(df[df['Churn'] == label]['Account length'], bins=30, alpha=0.55,
                    color=color, label='Churned' if label else 'Retained')
axes[1, 1].set_title('4) Account length (tenure) vs churn')
axes[1, 1].set_xlabel('Account length'); axes[1, 1].legend()

plt.tight_layout(); plt.show()

In [ ]:
# Supporting numbers for charts 2-4
print('Churn rate by International plan:')
print((df.groupby('International plan')['Churn'].mean() * 100).round(1))
print('\nMean total_charge / Account length by churn:')
display(df.groupby('Churn')[['total_charge', 'Account length']].mean().round(2))

> **Insights**
> - The base is imbalanced (~14% churn).
> - **International-plan** customers churn at a far higher rate — the strongest categorical driver.
> - Churners tend to have **higher total_charge** (heavy users on the wrong plan).
> - **Account length** barely differs between churners and stayers → tenure is a weak churn driver here (unlike IBM Telco).

## 4. Customer service calls — the strongest signal

Domain knowledge suggests repeated support contacts strongly predict churn. Let's confirm it.

In [ ]:
svc = df.groupby('Customer service calls')['Churn'].agg(['mean', 'count'])
svc.columns = ['churn_rate', 'customers']
display(svc)

ax = (svc['churn_rate'] * 100).plot(kind='bar', color='#dc2626')
ax.set_title('Churn rate by number of customer service calls')
ax.set_ylabel('Churn rate (%)'); ax.set_xlabel('Customer service calls')
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

> **Insight:** churn rate jumps sharply once a customer makes **4+** support calls — the basis for the `service_call_risk` engineered feature and the retention-escalation rule.

## 5. Numeric features by churn

In [ ]:
num_cols = ['Total day minutes', 'Total day charge', 'Total eve charge',
            'Total night charge', 'Total intl charge', 'Number vmail messages']
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.ravel(), num_cols):
    for label, color in [(0, '#16a34a'), (1, '#dc2626')]:
        ax.hist(df[df['Churn'] == label][col], bins=30, alpha=0.55, color=color,
                label='Churned' if label else 'Retained')
    ax.set_title(col); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
df.groupby('Churn')[num_cols].mean().T

## 6. Correlation & redundancy

Charge columns are a linear function of minutes (charge = minutes × rate), so they will be highly correlated. Tree models tolerate this; it's good to be aware of it.

In [ ]:
numeric = df.select_dtypes(include=[np.number])
corr = numeric.corr()

fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=90, fontsize=8)
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns, fontsize=8)
fig.colorbar(im, fraction=0.046, pad=0.04)
ax.set_title('Numeric feature correlation'); plt.tight_layout(); plt.show()

In [ ]:
corr['Churn'].drop('Churn').sort_values(ascending=False)

## 7. Key takeaways

- The data is **clean** (no missing values, no duplicates) but **imbalanced** (~14% churn) → handled with class weighting.
- **Customer service calls (4+)** is the single strongest churn driver.
- **International plan** subscribers churn far more often.
- **High daytime usage / charges** correlate with churn; **account length** does not.
- **Charge ≈ minutes** (perfectly collinear) → fine for tree models.

These observations motivate the engineered features in `feature_engineering.ipynb` and `src/feature_engineering.py`.